In [13]:
# Load env variables and create client
from dotenv import load_dotenv
load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-6" 

In [14]:
def add_user_message(messages, text):
    user_message = {"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages":  messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    message = client.messages.create(**params)  
    return message.content[0].text # type: ignore

In [3]:
messages = []

add_user_message(messages=messages, text="Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_016xG1yunYqWSBb71ui2E6df', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a 1 sentence description of a fake database:\n\n**"NomN', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='om Bites is a fictional database of over 5

In [ ]:
messages = []

add_user_message(messages=messages, text="Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    
    for text in stream.text_stream:
        #print(text, end="")
        pass
stream.get_final_message()

ParsedMessage(id='msg_01BCmpT5Nvv2PcdnmmXwU7Tx', container=None, content=[ParsedTextBlock(citations=None, text='Here is a one sentence description of a fake database:\n\n**"NutriTrack Pro Database"** — A fictional nutritional database containing detailed dietary information for over 500,000 food items, including made-up exotic ingredients like "Zorblefruit" and "Crystallized Moon Kelp," used for testing meal-planning application features.', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=79, server_tool_use=None, service_tier='standard'))


In [8]:
# controlling the output
messages = []

add_user_message(messages=messages, text="Generate a very short event bridge rul as json")
chat(messages)

'Here\'s a simple EventBridge rule in JSON:\n\n```json\n{\n  "Name": "my-rule",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["stopped"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "MySNSTopic",\n      "Arn": "arn:aws:sns:us-east-1:123456789012:my-topic"\n    }\n  ]\n}\n```\n\nThis rule:\n- **Triggers** when an EC2 instance is **stopped**\n- **Sends** a notification to an **SNS topic**'

In [27]:
#  controlling the output
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text # type: ignore

In [59]:
# controlling the output
messages = []

#add_assistant_message(messages, "```json")
system="Return only raw valid JSON. No intro text, no markdown, no explanation."
add_user_message(messages=messages, text="Generate a very short event bridge rule as json")
text=chat(messages=messages, system=system, stop_sequences=["```\n\n"])
text = text.replace("```json", "").replace("```", "").strip()
text

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}'

In [60]:
import json
json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}